In [1]:
!pip install dagshub mlflow -q
import dagshub, mlflow, pandas as pd, numpy as np
from kaggle_secrets import UserSecretsClient

# Setup Dagshub
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("DAGSHUB_TOKEN")
dagshub.init(repo_owner='kvizhmena11', repo_name='ML-assignment2-IEEE-Fraud-Detection', mlflow=True)

# Load Model
model = mlflow.sklearn.load_model("models:/XGBoost_Fraud_Pipeline/latest")
print("✅ Model loaded from Dagshub!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 87.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=26c6e964-8725-4f44-9e77-79a3a52d5d06&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3659fe538e71369c0bbf930ae193d57b6f4c3b5095f9f5e4719c8cf3f9a29dc9




Accessing as kvizhmena11

Initialized MLflow to track repo "kvizhmena11/ML-assignment2-IEEE-Fraud-Detection"

Repository kvizhmena11/ML-assignment2-IEEE-Fraud-Detection initialized!

✅ Model loaded from Dagshub!


In [5]:
# --- FINAL REPAIR BLOCK ---
base_path = '/kaggle/input/competitions/ieee-fraud-detection/'

# 1. Load Data
print("📂 Loading data...")
test_trans = pd.read_csv(base_path + 'test_transaction.csv')
test_id = pd.read_csv(base_path + 'test_identity.csv')
test_id.columns = [col.replace('-', '_') if 'id' in col else col for col in test_id.columns]
test = pd.merge(test_trans, test_id, on='TransactionID', how='left')
del test_trans, test_id

# 2. Basic Feature Engineering
emails = {'gmail': 'google', 'att.net': 'at&t', 'twc.com': 'spectrum', 'outlook.com': 'microsoft', 'msn.com': 'microsoft', 'hotmail.com': 'microsoft', 'live.com': 'microsoft', 'icloud.com': 'apple', 'mac.com': 'apple', 'me.com': 'apple'}
for c in ['P_emaildomain', 'R_emaildomain']:
    test[c + '_bin'] = test[c].map(emails)
    test[c + '_suffix'] = test[c].map(lambda x: str(x).split('.')[-1])

test['Trans_hour'] = (test['TransactionDT'] // 3600) % 24
test['TransactionAmt_log'] = np.log1p(test['TransactionAmt'])

# 3. FIX: Add the missing "KeyError" columns
print("🛠️ Fixing missing columns...")
for col in ['card1', 'card2', 'card3', 'card5']:
    if col in test.columns:
        test[f'{col}_freq'] = test[col].map(test[col].value_counts(normalize=True))

test['card1_TransactionAmt_mean'] = test.groupby('card1')['TransactionAmt'].transform('mean')
test['card1_TransactionAmt_std'] = test.groupby('card1')['TransactionAmt'].transform('std')
test['amt_vs_card_mean'] = test['TransactionAmt'] / (test['card1_TransactionAmt_mean'] + 1)

# 4. Predict
print("🔮 Processing categories and predicting...")
cols_to_use = model.feature_names_in_

# Create X_test
X_test = test.reindex(columns=cols_to_use, fill_value=0)

# FIX: Pipelines often hate 'category' type and strings. 
# We must manually encode any remaining strings to numbers.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in X_test.columns:
    if X_test[col].dtype == 'object' or X_test[col].dtype.name == 'category':
        # Fill missing values with 'nan' so LabelEncoder doesn't crash
        X_test[col] = le.fit_transform(X_test[col].astype(str))

# Now try to predict
try:
    preds = model.predict_proba(X_test)[:, 1]
except ValueError as e:
    print(f"❌ Still hitting a snag: {e}")
    # Plan B: If the pipeline is still picky, convert everything to float
    X_test = X_test.astype(np.float32)
    preds = model.predict_proba(X_test)[:, 1]

# 5. Save
submission = pd.DataFrame({'TransactionID': test['TransactionID'], 'isFraud': preds})
submission.to_csv('submission.csv', index=False)
print("✅ SUCCESS! submission.csv is created.")

📂 Loading data...
🛠️ Fixing missing columns...
🔮 Processing categories and predicting...
✅ SUCCESS! submission.csv is created.
